# 03 – Open-loop tracking of reflected signals

## What this notebook does

Notebook 01 tracks the **direct** signal of the zenith antenna in closed loop. A closed
loop cannot hold a weak, fading, partly diffuse **reflection**. This notebook does not use
a loop for the reflection. It **predicts** the delay, the Doppler and the carrier phase of
the reflection from a model, correlates the nadir channel at the predicted values, and
records the complex correlations. This is open-loop (OL) tracking.

| Item | Notebook 01 | Notebook 03 |
| --- | --- | --- |
| Antenna | Zenith RHCP | Nadir RHCP or LHCP |
| Tracking | Closed loop | Open loop (model-driven) |
| Needs | IQ file | IQ file, notebook 01 file, notebook 02 time (D-100), trajectory, attitude, lever arms, models |
| Writes | `tracking/*.h5` | `openloop/*.h5` (input of notebook 04, CWF, D-101) |

Plan: `docs/NB03_OPEN_LOOP_PLAN.md`. Decisions: D-087 to D-101. Known gaps and risks:
`docs/NB03_CONCERNS.md`. All logic is in `utils/openloop/`.

## Two modes

- `DATA_MODE = "synthetic"` (the default). **No reflection recording exists yet.** The
  notebook makes a zenith channel and a nadir channel from one known geometry
  (`utils/openloop/synthetic.py`) and runs every step against the truth. The same functions
  run on real data.
- `DATA_MODE = "real"`. The real-data path. It is written but **not yet run on a real
  recording**. It stops with a clear message while a value without a default (lever arms
  D-091, flight height D-099, water level D-093, time base D-100) is not set in
  `configs/openloop.yml`.

## Sign conventions

Code phase in ms of code grows with time. The carrier is $e^{+j2\pi\phi}$ with $\phi$ in
cycles. A longer path gives a smaller code phase and a smaller carrier phase. Doppler is
$f = -\frac{1}{\lambda}\frac{d\rho}{dt}$. These are the conventions of `tracking_channel`
and `tests/synthetic.py`; test `test_mode_a_model_matches_truth_and_open_loop_recovers_phase`
fixes them.

## 1. Imports

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import utils
from utils import tables
from utils.openloop import (
    acquisition, antennas, atmosphere, attitude, config, correlator, direct_ref, frames,
    heights, leakage, model, phase, reflectivity, sat_states, specular, surface,
    synthetic, timebase, trajectory,
)
from utils.openloop import io as olio
from utils.signal_interfaces import GpsL1CA, GpsL5

utils.plotting.setup_default_plotting()
C = frames.SPEED_OF_LIGHT

## 2. Configuration

### 2.1 Mode, signals and the config file

All tunable values are in `configs/openloop.yml` (CLAUDE.md rule 5). The few values in
this cell choose what to run. `SIGNAL_L1` and `SIGNAL_L5` are the two bands; the
dual-frequency filter (section 11) needs both on the same satellite.

In [ ]:
DATA_MODE = "synthetic"      # "synthetic" (default; no reflection data yet) or "real"
CONFIG_PATH = config.DEFAULT_PATH
SIGNAL_L1 = GpsL1CA
SIGNAL_L5 = GpsL5
PRN = 1
NADIR_POLARIZATION = "LHCP"  # "RHCP" or "LHCP"; one channel per run (D-041)

cfg = config.load_config(CONFIG_PATH)
tables.print_fields([
    ("data mode", DATA_MODE),
    ("config", str(CONFIG_PATH)),
    ("model mode", cfg["models"]["model_mode"]),
    ("orbits / specular", f'{cfg["models"]["orbits"]} / {cfg["models"]["specular"]}'),
    ("geoid / surface", f'{cfg["models"]["geoid"]} / water level {cfg["surface"]["water_level_m"]}'),
    ("ionosphere / troposphere", f'{cfg["models"]["ionosphere"]} / {cfg["models"]["troposphere"]}'),
])

### 2.2 The synthetic scenario

Used only when `DATA_MODE = "synthetic"`. One satellite on a straight-line track, one drone
above flat water. The drone moves up and down, so the excess Doppler is not zero, and moves
forward at 2 m/s. The receiver clock drifts by 0.3 ppm; in the direct-referenced model that
drift must cancel. The amplitudes give about 37 dB-Hz (L1) and 40 dB-Hz (L5) for the
reflection, and a direct-signal leak of 0.3 times the reflection amplitude on L1.

In [ ]:
SYN_DURATION_S = 1.5
SYN_FS_L1 = 4.0e6
SYN_FS_L5 = 20.46e6
SYN_WORLD = synthetic.SyntheticWorld(drone_height_m=30.0, vertical_amplitude_m=0.6,
                                     vertical_period_s=3.0, sat_elevation_deg=35.0)
SYN_REFLECTION_AMP = {"L1": 0.05, "L5": 0.03}
SYN_NOISE_SIGMA = 1.0
SYN_LEAK_AMP_L1 = 0.015
SYN_LEAK_PHASE_CYCLES = 0.21
SYN_TAU_HW_S = 30e-9          # nadir channel 30 ns later than zenith (to calibrate)
SYN_SEED = 7

### 2.3 The real-data inputs

Used only when `DATA_MODE = "real"`. The notebook 01 tracking file of the zenith channel
(same band), the nadir channel of the same collect, and the offset of the nadir file start
against the zenith file start. The four SDRs set their time at the same PPS edge (D-025),
but whether the recordings start on the same sample is **not confirmed** (concern C-4).

In [ ]:
TRACKING_FILE_L1 = None        # notebook 01 output for the L1-group zenith channel
TRACKING_FILE_L5 = None        # notebook 01 output for the L5-group zenith channel
EXPERIMENT_INDEX = 0
NADIR_CHANNEL_INDEX_L1 = None  # catalog index of the nadir channel (L1 group, chosen polarization)
NADIR_CHANNEL_INDEX_L5 = None
START_OFFSET_MS = 0.0          # the START_OFFSET_MS that notebook 01 used (uptime 0 there)
NADIR_MINUS_ZENITH_START_S = 0.0  # file start offset between the SDRs (C-4)
TRACK_SECONDS = 30.0

## 3. Load the inputs and line up the time

Real mode: the zenith tracking run gives $\tau_D$, $\phi_D$, $f_D$ and the data symbols
(`direct_ref.from_tracking_result`). The time base from notebook 02 maps the sample time
("uptime") to GPS time (D-100). Notebook 02 does not save its anchors yet (concern C-2),
so `time.timebase_file` in the config is a small YAML file written by hand from what
notebook 02 prints.

Synthetic mode: the generator gives the samples and a perfect direct reference (as if
notebook 01 had tracked with no error).

In [ ]:
bands = {}
if DATA_MODE == "synthetic":
    for band, signal_type, fs in (("L1", SIGNAL_L1, SYN_FS_L1), ("L5", SIGNAL_L5, SYN_FS_L5)):
        signal = signal_type(PRN)
        zen, nad, truth = synthetic.generate_pair(
            signal, SYN_WORLD, samp_rate=fs, duration_s=SYN_DURATION_S,
            reflection_amplitude=SYN_REFLECTION_AMP[band],
            leakage_amplitude=SYN_LEAK_AMP_L1 if band == "L1" else 0.0,
            leakage_phase_cycles=SYN_LEAK_PHASE_CYCLES,
            zenith_noise_sigma=SYN_NOISE_SIGMA, nadir_noise_sigma=SYN_NOISE_SIGMA,
            hardware_delay_s=SYN_TAU_HW_S, seed=SYN_SEED)
        bands[band] = dict(signal=signal, fs=fs, zenith=zen, nadir=nad, truth=truth,
                           direct=direct_ref.from_synthetic_truth(truth, signal),
                           file_start_uptime_ms=0.0)
    tb = SYN_WORLD.timebase()  # the world's receiver clock: uptime -> system time (D-105)
    span_ms = (20.0, SYN_DURATION_S * 1e3 - 20.0)
else:
    from utils import catalog, tracking_io
    from utils.environment_variables import get_collects_path

    config.require_real_data_values(cfg)
    tb = timebase.load_timebase(cfg["time"]["timebase_file"])
    for band, signal_type, tfile, ch in (("L1", SIGNAL_L1, TRACKING_FILE_L1, NADIR_CHANNEL_INDEX_L1),
                                         ("L5", SIGNAL_L5, TRACKING_FILE_L5, NADIR_CHANNEL_INDEX_L5)):
        if tfile is None or ch is None:
            continue
        run = tracking_io.load_tracking_results(tfile)
        signal = signal_type(PRN)
        result = run[signal.signal_id]
        channel = catalog.select_channel(get_collects_path(), EXPERIMENT_INDEX, ch)
        bands[band] = dict(signal=signal, fs=channel.samp_rate, channel=channel,
                           direct=direct_ref.from_tracking_result(result, signal),
                           # Uptime 0 of the tracking run is the sample at START_OFFSET_MS
                           # of the zenith file; the nadir file may start later (C-4).
                           file_start_uptime_ms=-START_OFFSET_MS + 1e3 * NADIR_MINUS_ZENITH_START_S)
    if not bands:
        raise ValueError("set TRACKING_FILE_L1/L5 and NADIR_CHANNEL_INDEX_L1/L5 in section 2.3")
    first = max(b["direct"].uptime_ms[0] for b in bands.values()) + 20.0
    span_ms = (first, first + TRACK_SECONDS * 1e3)

tables.print_fields([("bands", ", ".join(bands)), ("time base", tb.source),
                     ("span (uptime, ms)", f"{span_ms[0]:.1f} to {span_ms[1]:.1f}")])

## 4. Where the antennas were

Plan section 3.2. The trajectory gives one reference point (the rover antenna, Here3 now,
F9P later, D-087). The lever arms (D-091) and the attitude (D-089) move it to the zenith
and nadir phase centres:

$$\mathbf{R}_A = \mathbf{P} + C_n^e C_b^n(\mathbf{l}_A - \mathbf{l}_P), \qquad
\dot{\mathbf{R}}_A = \dot{\mathbf{P}} + C_n^e C_b^n\big(\boldsymbol\omega_{nb}^b\times(\mathbf{l}_A - \mathbf{l}_P)\big)$$

The phase-centre offset is a fixed vector along the boresight for each band (D-090).

In [ ]:
if DATA_MODE == "synthetic":
    def rx_tracks(t):
        return SYN_WORLD.nadir_ecef(t), None, SYN_WORLD.zenith_ecef(t), None
else:
    tcfg, acfg = cfg["trajectory"], cfg["attitude"]
    if tcfg["source"] == "static":
        traj = trajectory.static_trajectory(**tcfg["static"])
    else:
        traj = trajectory.read_rtklib_pos(tcfg["path"]).mask_quality(tuple(tcfg["accepted_quality"]))
    att = (attitude.read_ardupilot_csv(acfg["att_csv"], acfg["gps_csv"], acfg["message"])
           if acfg["source"] == "ardupilot_log" else attitude.constant_attitude(*acfg["constant_rpy_deg"]))
    ants = {name: antennas.AntennaGeometry(name, None if a["lever_arm_b"] is None else np.array(a["lever_arm_b"], float),
                                           np.array(a["boresight_b"], float), a["pco_m"])
            for name, a in cfg["antennas"].items()}
    antennas.require_lever_arms(ants)
    BAND_NAME = "L1"  # the PCO band; section 8 builds one model per band

    def rx_tracks(t):
        ref = ants[tcfg["reference_point"]]
        n_pos, n_vel = antennas.phase_centre_track(traj, att, ref, ants["nadir"], BAND_NAME, t)
        z_pos, z_vel = antennas.phase_centre_track(traj, att, ref, ants["zenith"], BAND_NAME, t)
        return n_pos, n_vel, z_pos, z_vel

t_plot = np.linspace(*span_ms, 200)
nadir_pos, _, zenith_pos, _ = rx_tracks(tb.gpst(t_plot))
lat, lon, h_nadir = frames.ecef_to_geodetic(nadir_pos)
_, _, h_zenith = frames.ecef_to_geodetic(zenith_pos)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_plot / 1e3, h_nadir - h_nadir[0], label="nadir APC")
ax.plot(t_plot / 1e3, h_zenith - h_nadir[0], label="zenith APC")
ax.set(xlabel="uptime (s)", ylabel="height change (m)", title="Antenna phase centres")
ax.legend(); plt.show()

## 5. Where the satellites were

Plan section 4. One interface for the broadcast (rapid) and the precise (SP3, high)
providers. The light time is solved separately for the direct path (to the zenith antenna)
and the reflected path (to the specular point). At UAS heights the satellite position
enters the excess path only through the elevation:
$\delta(\Delta\rho) \approx 2h\cos e\,\delta e$. A 1 m orbit error is about $10^{-5}$ m
of excess path at $h = 100$ m, so the **rapid** model is enough for Mode A.

In [ ]:
SAT_ID = f"G{PRN:02d}"
if DATA_MODE == "synthetic":
    provider = sat_states.TrackProvider(SYN_WORLD.sat_ecef)
else:
    from utils import broadcast_ephemeris
    t_mid = float(tb.gpst(np.mean(span_ms)))
    week, tow = int(t_mid // 604800), t_mid % 604800
    epoch = broadcast_ephemeris.datetime_from_gps(week, tow)
    ephs = broadcast_ephemeris.ephemerides_for([SAT_ID], tow, week, epoch)
    provider = sat_states.BroadcastProvider(ephs)
    if cfg["models"]["orbits"] == "precise":
        from utils import precise_orbits
        provider = sat_states.PreciseProvider(precise_orbits.load_sp3(epoch))

t_gps = tb.gpst(t_plot)
sat_pos, tau = sat_states.transmit_state(provider, SAT_ID, t_gps, zenith_pos)
el, az = frames.elevation_azimuth(zenith_pos, sat_pos)
d_e = 1.0 / np.linalg.norm(sat_pos - zenith_pos, axis=1)   # 1 m across the line of sight
height_now = cfg["flight"]["height_max_m"] if DATA_MODE == "real" else SYN_WORLD.drone_height_m
tables.print_fields([
    ("satellite", SAT_ID),
    ("elevation / azimuth", f"{np.degrees(el.mean()):.2f} deg / {np.degrees(az.mean()):.1f} deg"),
    ("transit time", f"{tau.mean() * 1e3:.3f} ms"),
    ("excess-path change for a 1 m orbit error", f"{np.max(2 * np.mean(height_now) * np.cos(el) * d_e):.1e} m"),
])

## 6. The reflecting surface and the specular point

Plan sections 5 and 6. The surface is geoid + water level (GEOID18 by default, D-092; the
ITRF-to-NAD83(2011) step is in `surface.GeoidGrid`). The **image** solution mirrors the
receiver in the plane tangent to the surface at the specular point (two passes); the
**iterative** solution follows Gleason (2019) on the curved surface, with a Newton-sized
step and the Snell residual as the stop test.

Finding in this draft: the two-pass image solution is exact to well under 1 mm at these
heights, because its second plane is tangent at the specular point itself. A single plane
through the point under the receiver misses by about $d^2/(2R_E)$ (the plan's 0.10 m at
$h = 100$ m, $e = 5^\circ$). See concern C-8.

In [ ]:
if DATA_MODE == "synthetic":
    surf = surface.Surface(surface.NoGeoid(), SYN_WORLD.surface_h_ell_m)
else:
    lat0, lon0 = np.degrees(lat.mean()), np.degrees(lon.mean())
    geoid = surface.make_geoid(cfg["models"]["geoid"], site_lat_deg=lat0, site_lon_deg=lon0,
                               constant_n_m=cfg["surface"]["constant_geoid_n_m"],
                               epoch_year=cfg["models"]["geoid_epoch_year"])
    surf = surface.Surface(geoid, cfg["surface"]["water_level_m"], cfg["models"]["use_deflection"])

img = specular.image_plane(sat_pos, nadir_pos, surf)
cur = specular.iterative_surface(sat_pos, nadir_pos, surf, tolerance_rad=1e-7)
wl = {b: C / d["signal"].carrier_freq_hz for b, d in bands.items()}
fz_b, fz_a = specular.fresnel_axes(wl["L1"], img.height_above_surface_m, img.elevation_rad)
fig, axs = plt.subplots(1, 2, figsize=(11, 3))
axs[0].plot(t_plot / 1e3, (img.reflected_path_m - cur.reflected_path_m) * 1e3)
axs[0].set(xlabel="uptime (s)", ylabel="image - iterative (mm)", title="Reflected path, two solvers")
axs[1].semilogy(t_plot / 1e3, cur.snell_residual_rad, label="iterative")
axs[1].semilogy(t_plot / 1e3, img.snell_residual_rad, label="image")
axs[1].set(xlabel="uptime (s)", ylabel="Snell residual (rad)"); axs[1].legend(); plt.show()
if DATA_MODE == "synthetic":
    # The Fermat truth uses the satellite at the receive time (no light time), so
    # compare the solver on that same satellite position.
    t_chk = t_plot[::40] / 1e3
    truth_path, _ = synthetic.truth_reflected_path_m(SYN_WORLD, t_chk)
    chk = specular.iterative_surface(SYN_WORLD.sat_ecef(t_chk), SYN_WORLD.nadir_ecef(t_chk), surf,
                                     tolerance_rad=1e-7)
    print("iterative minus Fermat truth (mm):", np.round((chk.reflected_path_m - truth_path) * 1e3, 3))
tables.print_fields([
    ("height above surface", f"{img.height_above_surface_m.mean():.2f} m"),
    ("elevation at the specular point", f"{np.degrees(img.elevation_rad.mean()):.2f} deg"),
    ("first Fresnel zone (L1)", f"{2 * fz_a.mean():.1f} m x {2 * fz_b.mean():.1f} m"),
    ("iterations (iterative solver)", f"{cur.iterations.max()}"),
])

### 6.1 Reflection coefficient and the expected level

Plan 9.1. Above the Brewster angle the reflection from water is mostly LHCP; below it,
mostly RHCP. The roughness loss is $e^{-(2k\sigma\sin e)^2}$.

In [ ]:
eps = complex(*cfg["surface"]["permittivity"])
e_axis = np.deg2rad(np.linspace(1, 90, 300))
r_rr, r_rl = reflectivity.fresnel_circular(eps, e_axis)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.degrees(e_axis), 20 * np.log10(np.abs(r_rr)), label="RHCP -> RHCP")
ax.plot(np.degrees(e_axis), 20 * np.log10(np.abs(r_rl)), label="RHCP -> LHCP")
ax.axvline(np.degrees(img.elevation_rad.mean()), color="k", ls=":", label=SAT_ID)
ax.set(xlabel="elevation (deg)", ylabel="|R|^2 (dB)", ylim=(-30, 0),
       title=f"Fresnel, eps = {eps}; Brewster ~ {np.degrees(reflectivity.brewster_elevation_rad(eps)):.1f} deg")
ax.legend(); plt.show()

## 7. The atmosphere on the two paths

Plan section 7. The ionosphere cancels in the excess path (no electrons under the drone),
so every ionosphere model returns 0 for the excess term. NeQuick-G (D-095) is not ported
yet (concern C-7); it matters only for the absolute mode. The troposphere does **not**
cancel: the reflected path crosses the bottom layer twice more. First order:

$$\Delta\rho_{trop} \approx 2\cdot10^{-6}N_0\,\frac{h}{\sin e}, \qquad \frac{\delta h}{h} \approx \frac{10^{-6}N_0}{\sin^2 e}$$

This first-order value is **not yet checked against ray tracing** (concern C-6).

In [ ]:
surface_h = float(np.mean(frames.ecef_to_geodetic(img.specular_ecef)[2]))
tropo_rapid = atmosphere.SaastamoinenTropo()
tropo = tropo_rapid
if cfg["models"]["troposphere"] == "gpt3":
    tropo = atmosphere.GPT3Tropo(cfg["models"]["gpt3_grid"])
elif cfg["models"]["troposphere"] == "none":
    tropo = atmosphere.NoTropo()
iono = atmosphere.NoIono()   # the excess term is zero for every model (plan 7.1)

n0 = tropo_rapid.surface_n0(0, 0, surface_h)
rows = []
for e_deg in (5, 10, 20, 30, 60):
    for h in (10, 30, 60, 120):
        d = atmosphere.excess_tropo_first_order_m(n0, h, np.deg2rad(e_deg))
        rows.append((e_deg, h, d, d / (2 * np.sin(np.deg2rad(e_deg)))))
print(f"surface refractivity N0 = {float(n0):.0f} (standard atmosphere at {surface_h:.0f} m)")
print(" elev  h(m)  excess tropo (m)  height bias if ignored (m)")
for e_deg, h, d, dh in rows:
    print(f"{e_deg:5d} {h:5d} {d:16.3f} {dh:27.3f}")

## 8. Producing the model

Plan section 8.1, Mode A (direct-referenced):

$$\tilde\chi_R = \chi_D - 10^3\left(\frac{D}{c}+\tau_{hw}\right),\quad
\tilde\phi_R = \phi_D - \frac{D}{\lambda} + \phi_{hw},\quad
\tilde f_R = f_D - \frac{\dot D}{\lambda},\qquad D = \Delta\rho + \Delta\rho_{trop}$$

The geometry is evaluated at `openloop.model_rate_hz`; $D$ is a cubic spline in uptime,
and its derivative gives the Doppler.

In [ ]:
anchor_step_ms = 1e3 / cfg["openloop"]["model_rate_hz"]
anchors = np.arange(span_ms[0] - 10.0, span_ms[1] + 10.0 + anchor_step_ms, anchor_step_ms)
models = {}
for band, d in bands.items():
    if DATA_MODE == "real":
        BAND_NAME = band
    tau_hw = cfg["calibration"]["tau_hw_ns"].get(f"{band}_{NADIR_POLARIZATION}", 0.0) * 1e-9
    geo = model.compute_geometry(sat_id=SAT_ID, provider=provider, rx_tracks=rx_tracks, surface=surf,
                                 timebase=tb, uptime_ms=anchors, solver=cfg["models"]["specular"],
                                 troposphere=tropo, ionosphere=iono)
    models[band] = model.ReflectionModel(d["direct"], geo, d["signal"].carrier_freq_hz, tau_hw_s=tau_hw)

m1 = models["L1"]
u = np.linspace(*span_ms, 400)
fig, axs = plt.subplots(1, 3, figsize=(13, 3))
axs[0].plot(u / 1e3, m1.excess_m(u)); axs[0].set(xlabel="uptime (s)", ylabel="D (m)", title="Excess path")
axs[1].plot(u / 1e3, m1.doppler_hz(u) - m1.direct.doppler_at(u))
axs[1].set(xlabel="uptime (s)", ylabel="f_R - f_D (Hz)", title="Excess Doppler (L1)")
axs[2].plot(u / 1e3, (m1.direct.code_phase_at(u) - m1.code_phase_ms(u)) * 1e-3 * SIGNAL_L1.tracking_code_rate_chips_per_sec)
axs[2].set(xlabel="uptime (s)", ylabel="chips", title="Reflected delay behind direct (L1)")
plt.tight_layout(); plt.show()

if DATA_MODE == "synthetic":
    tr = bands["L1"]["truth"]
    err = m1.geometry.excess_geometric_m - np.interp(m1.geometry.uptime_ms, tr.t_s * 1e3, tr.excess_path_m)
    print(f"geometric excess path, model minus truth: max {np.max(np.abs(err)) * 1e3:.2f} mm")
    print(f"troposphere excess in the model: {m1.geometry.excess_tropo_m.mean() * 1e3:.1f} mm "
          "(the synthetic signal has none, so this shows as a bias later)")

### 8.1 Error budget for this geometry

Plan 8.6. The row values are for the current mean height and elevation.

In [ ]:
h_mean, e_mean = float(np.mean(m1.geometry.height_m)), float(np.mean(m1.geometry.specular.elevation_rad))
rows = [("vertical position 2 cm (RTK/PPK fix)", 2 * 0.02 * np.sin(e_mean)),
        ("vertical position 5 m (SPP)", 2 * 5.0 * np.sin(e_mean)),
        ("attitude 1 deg x lever arm 0.3 m", 0.3 * np.deg2rad(1.0)),
        ("orbit 1 m (broadcast)", 2 * h_mean * np.cos(e_mean) / 2.0e7),
        ("troposphere not modelled", float(np.mean(m1.geometry.excess_tropo_m)))]
print(f"h = {h_mean:.1f} m, e = {np.degrees(e_mean):.1f} deg")
for name, value in rows:
    print(f"  {name:40s} {value:10.4f} m of excess path")
for band, sig in (("L1", SIGNAL_L1), ("L5", SIGNAL_L5)):
    chip = C / sig.tracking_code_rate_chips_per_sec
    print(f"  {band}: a quarter chip is {chip / 4:.1f} m; 2 h_max is {2 * np.max(m1.geometry.height_m):.1f} m")

### 8.2 Mode B cross-check (absolute model)

Plan 8.2. Mode B predicts the code phase from the geometry and the clocks alone (D-105):

$$\chi_B = 10^3\left(t - \frac{\rho_R + I_R + T_R}{c} - \tau_{grp} + \delta t_{tx}\!\left(t - \frac{\rho_R}{c}\right)\right)$$

with $t$ the system time of the samples. The receiver clock enters only through the time
base, $t = T - \delta t_{rx}$ (notebook 02's clock series on real data); it is not added a
second time. Mode A minus Mode B must be a constant (the unknown start value of the
tracking code phase). On real data the full ionosphere of the path is not modelled
here, so its slow change (centimetres per minute) and the receiver-clock noise show in
the difference.

In [ ]:
# Mode B needs the FULL delays of the reflected path, not the excess (D-105):
# troposphere of the direct path at the zenith antenna + the excess troposphere.
lat_z, _, h_z = frames.ecef_to_geodetic(m1.geometry.zenith_ecef)
tropo_reflected_m = (tropo.delay_m(np.deg2rad(m1.geometry.elevation_deg), float(np.mean(lat_z)),
                                   float(np.mean(h_z)))
                     + m1.geometry.excess_tropo_m)
# tb: gpst() is system time and already holds the receiver clock (dt_rx = T - t).
mode_b = model.absolute_code_phase_ms(m1.geometry, tb, provider, SAT_ID,
                                      reflected_tropo_m=tropo_reflected_m)
diff_m = (m1.code_phase_ms(m1.geometry.uptime_ms) - mode_b) * 1e-3 * C
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(m1.geometry.uptime_ms / 1e3, (diff_m - diff_m.mean()) * 1e3)
ax.set(xlabel="uptime (s)", ylabel="mm", title="Mode A minus Mode B (mean removed)"); plt.show()

## 9. Acquiring the reflection

Plan section 9. A small delay-Doppler map around the model, incoherent sum of coherent
sums. The noise floor comes from lags before the direct leakage. The peak lag measures the
hardware delay $\tau_{hw}$ (the model does not know it yet). `acquisition.peak_method` sets how
the peak is refined between the grid lags (D-103):

- `parabola` (default): through the top three lags. It looks only at the tip, so the direct
  leakage near the peak (C-9) moves it less. For the sharp L1 C/A triangle it is biased by up
  to about 0.1 of the lag step (0.004 chip on the 0.05-chip grid here), so keep the step fine.
- `early_late`: $A(x-d) = A(x+d)$. Exact for any symmetric peak, but a second path near the
  peak (leakage, diffuse scattering) pulls it. In this synthetic run the L1 leakage moves it
  by about 11 ns.

In [ ]:
oc = cfg["openloop"]
def lags_for(signal):
    chip = C / signal.tracking_code_rate_chips_per_sec
    h_max = cfg["flight"]["height_max_m"] or float(np.max(m1.geometry.height_m)) + 5.0
    return correlator.lag_window_for_height(h_max, chip, oc["lag_margin_chips"], oc["lag_step_chips"],
                                            tuple(oc["noise_lags_chips"]))

def run_band(band, lags, doppler_offsets=None, span=None):
    d = bands[band]
    lo, hi = span or span_ms
    if DATA_MODE == "synthetic":
        return correlator.run_open_loop(d["nadir"], samp_rate=d["fs"], start_uptime_ms=0.0,
                                        signal=d["signal"], model=models[band], lags_chips=lags,
                                        doppler_offsets_hz=doppler_offsets,
                                        first_uptime_ms=lo, last_uptime_ms=hi)
    ch = d["channel"]
    return correlator.run_open_loop_file(ch.filepath, ch.sample_params, samp_rate=d["fs"],
                                         file_start_uptime_ms=d["file_start_uptime_ms"],
                                         signal=d["signal"], model=models[band], lags_chips=lags,
                                         doppler_offsets_hz=doppler_offsets,
                                         first_uptime_ms=lo, last_uptime_ms=hi)

ac = cfg["acquisition"]
acq_span = (span_ms[0], span_ms[0] + ac["incoherent_blocks"] * oc["coherent_ms"])
ddms = {}
for band in bands:
    sig = bands[band]["signal"]
    lags = lags_for(sig)
    ol_acq = run_band(band, lags, np.array(ac["doppler_offsets_hz"]), acq_span)
    comp = 0
    wiped = phase.wipe(ol_acq, sig, bands[band]["direct"], comp)
    ddms[band] = acquisition.delay_doppler_map(
        ol_acq, wiped=wiped, noise_chips=tuple(oc["noise_lags_chips"]),
        coherent_blocks=int(oc["coherent_ms"]), p_fa_total=ac["p_fa_total"],
        search_chips=(-oc["lag_margin_chips"], lags[-1]),
        peak_method=ac["peak_method"], peak_half_spacing_chips=ac["peak_half_spacing_chips"])

fig, axs = plt.subplots(1, len(ddms), figsize=(6 * len(ddms), 3))
for ax, (band, ddm) in zip(np.atleast_1d(axs), ddms.items()):
    im = ax.imshow(10 * np.log10(ddm.power / ddm.noise_power), aspect="auto", origin="lower",
                   extent=[ddm.lags_chips[0], ddm.lags_chips[-1], -0.5, len(ddm.doppler_offsets_hz) - 0.5])
    ax.set_yticks(range(len(ddm.doppler_offsets_hz)), [f"{f:+.0f}" for f in ddm.doppler_offsets_hz])
    ax.set(xlabel="excess-delay lag (chips)", ylabel="Doppler offset (Hz)",
           title=f"{band}: SNR {ddm.snr_db:.1f} dB, {'detected' if ddm.detected else 'NOT detected'}")
    plt.colorbar(im, ax=ax, label="dB over noise")
plt.tight_layout(); plt.show()

for band, ddm in ddms.items():
    rate = bands[band]["signal"].tracking_code_rate_chips_per_sec
    tau_est, _ = acquisition.estimate_tau_hw_s([ddm.peak_lag_chips], rate)
    extra = f" (synthetic truth {SYN_TAU_HW_S * 1e9:.1f} ns)" if DATA_MODE == "synthetic" else ""
    print(f"{band}: peak lag {ddm.peak_lag_chips:+.3f} chips -> tau_hw = {tau_est * 1e9:.1f} ns{extra}")
    models[band].tau_hw_s += tau_est   # use the calibration from here on (plan 9.4)

## 10. Tracking in open loop

Plan section 10.1. Each block is one code period of the reflected replica, so a block
never straddles a data or overlay transition. The waveforms are kept raw for notebook 04.
For L5 the pilot (L5Q) is used for the phase; its NH20 overlay offset is searched once.

In [ ]:
ol_runs = {}
for band in bands:
    ol_runs[band] = run_band(band, lags_for(bands[band]["signal"]))
    print(f"{band}: {ol_runs[band].num_blocks} blocks, {len(ol_runs[band].lags_chips)} lags, "
          f"components {ol_runs[band].component_names}")

PHASE_COMPONENT = {"L1": 0, "L5": 1}   # L1 C/A data; L5Q pilot
wiped_runs, lag_idx = {}, {}
for band, ol in ol_runs.items():
    sig, comp = bands[band]["signal"], PHASE_COMPONENT[band]
    j0 = int(np.argmin(np.abs(ol.lags_chips)))
    off = phase.search_overlay_offset(ol, sig, bands[band]["direct"], comp, j0)
    wiped_runs[band] = phase.wipe(ol, sig, bands[band]["direct"], comp, off)
    lag_idx[band] = j0
    print(f"{band}: overlay offset {off} periods, reflection lag index {j0}")

ol = ol_runs["L1"]
z = acquisition.coherent_sum(wiped_runs["L1"][:, 0, :], int(oc["coherent_ms"]))
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.imshow(np.abs(z).T, aspect="auto", origin="lower",
          extent=[ol.uptime_ms[0] / 1e3, ol.uptime_ms[-1] / 1e3, ol.lags_chips[0], ol.lags_chips[-1]])
ax.set(xlabel="uptime (s)", ylabel="excess-delay lag (chips)", title=f"L1 {NADIR_POLARIZATION}: |waveform|, 20 ms coherent")
plt.show()

## 11. Phase

Plan section 10.2 to 10.4. Residual phase at the reflection lag, C/N0 per coherent sum,
the coherence metric $\kappa$, then the adapted SCANF filter on L1 and L5 together. The
measured excess path is $\hat D = \tilde D + r$.

In [ ]:
pc = cfg["phase"]
noise_masks = {b: acquisition.noise_lag_mask(o.lags_chips, tuple(oc["noise_lags_chips"])) for b, o in ol_runs.items()}
series = {b: phase.coherent_series(ol_runs[b], wiped_runs[b], lag_idx[b], int(oc["coherent_ms"]), noise_masks[b])
          for b in ol_runs}
kappa = {b: phase.coherence_metric(s.prompt, int(pc["coherence_window"])) for b, s in series.items()}
if "L5" in series:
    s1, s5 = phase.align_series(series["L1"], series["L5"])
    filt = phase.scanf([s1, s5], process_accel_mps2=pc["process_accel_mps2"],
                       slip_search=int(pc["slip_search"]), min_cn0_dbhz=pc["min_cn0_dbhz"])
else:
    s1 = series["L1"]
    filt = phase.scanf([s1], process_accel_mps2=pc["process_accel_mps2"],
                       slip_search=int(pc["slip_search"]), min_cn0_dbhz=pc["min_cn0_dbhz"])

fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for b, s in series.items():
    axs[0].plot(s.uptime_ms / 1e3, s.cn0_dbhz, label=b)
    axs[1].plot(s.uptime_ms / 1e3, kappa[b], label=b)
axs[0].set(ylabel="C/N0 (dB-Hz)"); axs[0].legend()
axs[1].axhline(pc["coherence_min"], color="k", ls=":"); axs[1].set(ylabel="kappa")
axs[2].plot(filt.uptime_ms / 1e3, filt.measured_m - filt.measured_m[0], ".", ms=2, alpha=0.4)
axs[2].plot(filt.uptime_ms / 1e3, filt.residual_m - filt.residual_m[0], "k", label="SCANF r")
axs[2].set(xlabel="uptime (s)", ylabel="r (m)"); axs[2].legend()
plt.tight_layout(); plt.show()
print("slips found per band:", filt.slip_events.sum(axis=0))

## 12. Observables

Plan section 11.1. The carrier observable gives the height correction $\delta h$ from
$\hat D - \tilde D = 2\,\delta h \sin e + b$. In the synthetic run the true height is known,
so the only expected bias is the troposphere term that the model adds and the synthetic
signal does not have (section 7).

In [ ]:
geo1 = models["L1"].geometry
t_f = filt.uptime_ms
e_at = np.interp(t_f, geo1.uptime_ms, geo1.specular.elevation_rad)
model_ex = np.interp(t_f, geo1.uptime_ms, geo1.excess_total_m)
measured_ex = filt.excess_path_m          # D^ = D~ + r, up to the arc constant
if DATA_MODE == "synthetic":
    tr = bands["L1"]["truth"]
    truth_ex = np.interp(t_f, tr.t_s * 1e3, tr.excess_path_m)
    err = (measured_ex - truth_ex) - np.mean(measured_ex - truth_ex)
    tables.print_fields([
        ("measured minus true excess path (arc mean removed)", f"RMS {np.sqrt(np.mean(err**2)) * 1e3:.2f} mm, "
                                                                 f"max {np.max(np.abs(err)) * 1e3:.2f} mm"),
        ("arc constant (ambiguity + reflection phase + tropo model)", f"{np.mean(measured_ex - truth_ex) * 1e3:.1f} mm"),
    ])
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t_f / 1e3, err * 1e3)
    ax.set(xlabel="uptime (s)", ylabel="mm", title="Measured minus true excess path (L1+L5 SCANF)"); plt.show()

e_span = np.degrees(np.ptp(e_at))
if e_span > 2.0:
    fit = heights.fit_height([{"t_s": t_f / 1e3, "residual_m": filt.residual_m, "elevation_rad": e_at}])
    tables.print_fields([("height correction dh", f"{fit.dh_m * 100:.2f} +/- {fit.sigma_dh_m * 100:.2f} cm"),
                         ("condition number", f"{fit.condition:.1f}")])
else:
    print(f"The elevation changes by only {e_span:.3f} deg in this arc, so the height correction and the "
          "arc constant cannot be separated (plan 11.1). Use a longer arc or several satellites.")

### 12.1 Direct-signal leakage (estimate and flags only, D-098)

Plan 9.3. The zenith channel is correlated with the same reflected replica; the least-
squares coupling $\hat\beta$ is estimated per window. Its phase follows the B200-to-B210
LO phase, so a flat phase plot is the real-data check of D-097. The `diversity` value
bounds the bias that the reflection itself puts into $\hat\beta$ (concern C-9).

In [ ]:
lc = cfg["leakage"]
band = "L1"
d = bands[band]
if DATA_MODE == "synthetic":
    ol_zen = correlator.run_open_loop(d["zenith"], samp_rate=d["fs"], start_uptime_ms=0.0, signal=d["signal"],
                                      model=models[band], lags_chips=ol_runs[band].lags_chips,
                                      first_uptime_ms=span_ms[0], last_uptime_ms=span_ms[1])
else:
    raise NotImplementedError("real mode: run correlator.run_open_loop_file on the zenith file here (C-4)")
zn = wiped_runs[band][:, 0, lag_idx[band]]
zz = phase.wipe(ol_zen, d["signal"], d["direct"], 0)[:, 0, lag_idx[band]]
n = min(len(zn), len(zz))
est = leakage.estimate_beta(zn[:n], zz[:n], ol_runs[band].uptime_ms[:n], ol_runs[band].model_excess_m[:n],
                            wl[band], window=int(lc["window_blocks"]) * int(oc["coherent_ms"]),
                            min_turns=lc["min_turns"], strong_fraction=lc["strong_fraction"])
fig, axs = plt.subplots(1, 2, figsize=(11, 3))
axs[0].plot(est.uptime_ms / 1e3, np.abs(est.beta), "o-"); axs[0].set(xlabel="uptime (s)", ylabel="|beta|")
axs[1].plot(est.uptime_ms / 1e3, np.angle(est.beta) / (2 * np.pi), "o-")
axs[1].set(xlabel="uptime (s)", ylabel="arg beta (cycles)", title="Flat = LO phase stable (D-097)")
plt.tight_layout(); plt.show()
if DATA_MODE == "synthetic":
    print(f"truth: |beta| = {SYN_LEAK_AMP_L1:.3f} (leak amplitude / zenith amplitude 1), "
          f"arg beta = {SYN_LEAK_PHASE_CYCLES:.3f} cycles")
print("separable windows:", est.separable.sum(), "of", len(est.separable),
      "| strong-leak windows:", est.strong.sum(), "| diversity:", np.round(est.diversity, 2))

## 13. Save the results

`<OUTPUTS_PATH>/openloop/<name>.h5`: every raw waveform, the model, and the products, with
the model choices as provenance (plan 11.2).

In [ ]:
from utils.environment_variables import get_outputs_path

name = f"nb03_{DATA_MODE}_{SAT_ID}_{NADIR_POLARIZATION}.h5"
out_path = get_outputs_path() / "openloop" / name
products = {f"{band}": {} for band in ol_runs}
for band, s in series.items():
    products[band] = {"cn0_dbhz": s.cn0_dbhz, "wrapped_rad": s.wrapped_rad, "coherence": kappa[band],
                      "phase_uptime_ms": s.uptime_ms}
products["L1"].update({"scanf_residual_m": filt.residual_m, "scanf_uptime_ms": filt.uptime_ms,
                       "leak_beta_re": est.beta.real, "leak_beta_im": est.beta.imag})
olio.save(out_path, {f"{SAT_ID}_{b}": o for b, o in ol_runs.items()},
          provenance={"data_mode": DATA_MODE, "models": cfg["models"], "polarization": NADIR_POLARIZATION,
                      "tau_hw_s": {b: m.tau_hw_s for b, m in models.items()}, "time_base": tb.source},
          products={f"{SAT_ID}_{b}": p for b, p in products.items()})
print("wrote", out_path)

## Exercises

1. Set `SYN_WORLD.sat_elevation_deg = 8` and `drone_height_m = 100`. Compare the image and
   iterative solvers, and read the troposphere table again.
2. Raise `SYN_NOISE_SIGMA` until the L1 phase starts to slip. How many slips does the
   single-frequency filter find, and how many does the L1 + L5 filter find?
3. Set `SYN_LEAK_AMP_L1` to the reflection amplitude. How does the waveform change? When
   does the leakage estimate become unreliable (look at `diversity`)?

## Not covered here

- Removal of the direct-signal leakage (later work, D-098).
- The NeQuick-G electron-density model (C-7), the GPT3/VMF3 mapping functions (C-6).
- A ray-traced troposphere for the excess path (C-6).
- CWF (notebook 04, D-101), and a learned retracker (Yang 2025).
- The pipeline window (plan phase 9).